# ₿ Bitcoin Puzzle #71 — RTX PRO 6000 Blackwell ile solo, rastgele GPU araması

Bu notebook her şeyi kendisi yapar: **Çalışma zamanı → Tümünü çalıştır** (Runtime → Run all) demen yeterli.

1. GPU'yu ve hedef adresin bakiyesini kontrol eder. Bulmaca çözülmüşse durur.
2. [VanitySearch-Blackwell](https://github.com/StarWWW/VanitySearch-Blackwell) deposunu (GPLv3) klonlar ve **bu GPU'nun mimarisi için** derler (Blackwell = `sm_120`). Depo, FixedPaul'un [VanitySearch-Bitcrack](https://github.com/FixedPaul/VanitySearch-Bitcrack) aracının sm_120 için ayarlanmış hâlidir. Kendi `-random` modu var.
   * **Hız:** Orijinale göre anahtar başına **%10.8 daha az GPU çevrimi** harcıyor (Nsight Compute, sabit saat). RTX PRO 6000'de 8625 → 9479 MK/s (+%9.9). Güç limitli bir laptopta (RTX 5050, 80 W) +%9.0 ölçüldü.
   * **Hata düzeltmeleri:** GPU çıktı tamponu taşması, duraklat/devam ve `-random` hataları, güvenli Ctrl+C. Ayrıntılar deponun README'sinde.
   * Kaynak, hücre 1'deki `VS_REPO` / `VS_REF` ile seçilir (yerel bir klonun yolu da olur). Aynı commit aynı ayarlarla bir kez derlenir, sonraki çalıştırmalar önbellekteki aracı kullanır. Öz-testi geçemezse notebook, aynı deponun geçmişindeki orijinal FixedPaul kaynağına döner.
3. **Öz-test:** Rastgele bir test anahtarı saklar ve aracın onu gerçekten bulduğunu doğrular (birkaç saniye). Geçemezse arama başlamaz.
4. **Arama:** `2^70 … 2^71-1` aralığında rastgele arar. Araç bir anahtar bildirdiğinde Python'da bağımsız olarak doğrulanır, sonra kayıt klasörüne (Colab'da Drive) ve ikinci bir yedek dosyaya yazılır. Araç çökerse otomatik olarak yeniden başlatılır (art arda 5 kez GPU'ya ulaşamadan kapanırsa durup çıktıyı gösterir). İstatistikler oturumlar arasında kayıt klasöründe birikir.

**Başlamadan önce:** *Çalışma zamanı → Çalışma zamanı türünü değiştir* menüsünden **RTX PRO 6000 (Blackwell)** GPU'yu seç. Drive izni istendiğinde onay ver.


### 📜 Çoklu adres modu (`SEARCH_MODE = "multi"`)

Tek bir bulmaca adresi yerine bir **adres listesi** aratabilirsin. Hücre 1'de `SEARCH_MODE = "multi"` seç, `ADDRESS_LIST` ile listeyi (`p2pkh` / `bech32`) belirt. Notebook listeyi `tools/fetch_addresses.py` ile [pymmdrza/Bitcoin_Wallet_Addresses](https://huggingface.co/datasets/pymmdrza/Bitcoin_Wallet_Addresses) datasetinden indirir (dataset **gated**: `HF_TOKEN` ayarını doldur ve dataset sayfasında erişimi kabul et). Kendi dosyan için `ADDRESS_LIST = "custom"` + `ADDRESS_FILE`. Her satırda bir adres; hepsi aynı türde olmalı. GPU bunları bir bloom filtresiyle tarar, eşleşen her anahtar Python'da bağımsız doğrulanır. Öz-test de bu modda tek adres yerine bloom filtresi yolunu, listenin adres türüyle test eder. **Not:** rastgele bir aralıkta fonlu cüzdan araması gerçekçi bir bulma şansı taşımaz (anahtarlar 2^256 üzerinde düzgün dağılır); bu özellik filtreyi ölçeklemek ve kendi zayıf anahtarlarını denemek içindir.
---

### ⚠️ Gerçekçi beklenti (lütfen oku)

Aralıkta 2^70 ≈ **1.18 × 10²¹** anahtar var. Public key açıkta olmadığı için Kangaroo/BSGS kullanılamıyor; tek yol kaba kuvvet. Tek bir RTX PRO 6000 ile tahmini hız **~9–10 Gkey/s**:

| Süre | Bulma olasılığı (10 Gkey/s) |
|---|---|
| 1 saat | ≈ 1 / 33 milyon |
| 1 gün | ≈ 1 / 1.37 milyon |
| 1 ay | ≈ 1 / 45 bin |
| Tüm aralık | ≈ **3.700 yıl** |

Ödül 7.1 BTC. Bu ödülle bir günlük GPU çalışmasının beklenen değeri yaklaşık **520 satoshi** (~0.0000052 BTC). Bu, Colab'da bir günlük GPU ücretinin çok altında kalır. Bunu bir piyango gibi düşün ve kaybetmeyi göze alabileceğin kadar compute harca.

**Colab politikası:** Colab'ın SSS sayfası "kripto para madenciliği" ve "şifre kırma" gibi işleri yönetilen çalışma zamanlarında yasaklı faaliyetler arasında sayıyor. Bu iş bunlara çok yakın görülebilir, oturum kapatma veya hesap kısıtlaması riski var. Bu riski sen değerlendir. Notebook Colab'a özel değildir, aşağıdaki bölüme bak.

### 💻 Colab dışında çalıştırma (yerel PC, kiralık GPU sunucusu, Kaggle, Docker…)

Aynı notebook değişiklik yapmadan çalışır; ortamı kendisi algılar. Gerekenler:

| | Linux (Ubuntu/Debian, WSL2, RunPod, Vast.ai…) | Windows 10/11 |
|---|---|---|
| GPU | NVIDIA GPU + sürücü (`nvidia-smi` çalışmalı) | NVIDIA GPU + sürücü |
| Derleyici | `git`, `make`, `g++`; root/parolasız `sudo` varsa eksikler apt ile kurulur | [Git](https://git-scm.com/download/win) + **Visual Studio 2022 Build Tools** ("C++ ile masaüstü geliştirme") |
| CUDA | Toolkit (Blackwell için ≥ 12.8). Yoksa ve apt + yetki varsa `AUTO_INSTALL_CUDA` uygun `cuda-nvcc` sürümünü kurar | [CUDA Toolkit](https://developer.nvidia.com/cuda-downloads) (Blackwell için ≥ 12.8) |
| Python | 3.8+ ve Jupyter (`pip install notebook`) ya da VS Code | aynı |

* `#@param` satırları Colab dışında normal Python atamasıdır; ayarları hücre 1'de elle değiştir.
* Google Drive yalnızca Colab'da kullanılır. Başka yerde kayıtlar (istatistik, log, **bulunan anahtar**) ve derlenen araç `~/vanitysearch-puzzle/` klasörüne yazılır (Windows: `%USERPROFILE%\vanitysearch-puzzle`). Farklı bir yer için Jupyter'i `PUZZLE_HOME` ortam değişkeniyle başlat. Bu klasör bilerek repo dışındadır: bulunan anahtar yanlışlıkla bir commit'e girmesin.
* Birden çok GPU varsa hücre 1'deki `GPU_ID` ile seç (`nvidia-smi` sırası). Her GPU için ayrı bir notebook/çekirdek çalıştırabilirsin.
* Desteklenmeyenler: macOS (NVIDIA CUDA yok) ve ARM işlemciler (aracın CPU tarafı x86-64 SSE kodu içeriyor).

**Tek GPU = tek süreç:** Aynı GPU'da iki arama aynı anda çalıştırılmaz. Notebook GPU'nun tamamını tek araca verir.


In [ ]:
#@title ⚙️ 1) Ayarlar
#@markdown **Arama modu:** `puzzle` = tek bir bulmaca adresi · `multi` = bir adres listesi dosyası (indiriciden ya da kendi dosyan)
SEARCH_MODE    = "puzzle"   #@param ["puzzle", "multi"]

#@markdown --- `puzzle` modu ---
TARGET_ADDRESS = "1PWo3JeB9jrGwfHDNpdGK54CRas7fsVzXU"  #@param {type:"string"}
PUZZLE_BITS    = 71         #@param {type:"integer"}

#@markdown --- `multi` modu (çoklu adres) ---
#@markdown İndirilecek liste (`tools/fetch_addresses.py`). `custom` = kendi dosyanı `ADDRESS_FILE` ile ver.
ADDRESS_LIST   = "p2pkh"    #@param ["p2pkh", "bech32", "custom"]
#@markdown Adres dosyası yolu. Boşsa listeye göre otomatik ad verilip indirilir (büyük dosya, Drive'a değil yerel diske).
ADDRESS_FILE   = ""         #@param {type:"string"}
#@markdown Hugging Face erişim jetonu (dataset gated). Boşsa `HF_TOKEN` ortam değişkeni kullanılır.
HF_TOKEN       = ""         #@param {type:"string"}
#@markdown `multi` modunda taranacak aralık: başlangıç anahtarı (HEX) ve bit genişliği.
MULTI_START    = "1"        #@param {type:"string"}
MULTI_RANGE_BITS = 71       #@param {type:"integer"}

#@markdown --- ortak ---
#@markdown **VanitySearch stratejisi:** `native` = aracın kendi `-random` modu · `blocks` = rastgele 2^BLOCK_BITS bloklar (Drive'a kaydedilir, tekrar taranmaz).
VS_STRATEGY    = "native"   #@param ["native", "blocks"]
BLOCK_BITS     = 42         #@param {type:"integer"}
USE_DRIVE      = True       #@param {type:"boolean"}
#@markdown **Kaynak:** VanitySearch-Blackwell deposu (GitHub adresi ya da yerel bir klonun yolu) ve dalı, etiketi ya da commit'i.
VS_REPO        = "https://github.com/StarWWW/VanitySearch-Blackwell.git"  #@param {type:"string"}
VS_REF         = "main"     #@param {type:"string"}
AUTO_INSTALL_CUDA = True    #@param {type:"boolean"}
#@markdown Birden çok GPU'lu makinede kullanılacak GPU'nun numarası (`nvidia-smi` sırası).
GPU_ID         = 0          #@param {type:"integer"}

MULTI = (SEARCH_MODE == 'multi')
if MULTI:
    RANGE_BITS  = int(MULTI_RANGE_BITS)
    RANGE_START = int(MULTI_START, 16)
    PUZZLE_BITS = RANGE_BITS                        # kayıt/etiket adları için
else:
    RANGE_BITS  = PUZZLE_BITS - 1                   # 71. bulmaca: anahtar 2^70 ... 2^71-1 aralığında
    RANGE_START = 1 << RANGE_BITS
assert RANGE_BITS >= 34, 'GPU araçları çok küçük aralıklarda çalışmaz; en az 34 bitlik bir aralık kullan.'
RANGE_END   = RANGE_START + (1 << RANGE_BITS) - 1
RANGE_SIZE  = RANGE_END - RANGE_START + 1
if MULTI and VS_STRATEGY == 'blocks' and BLOCK_BITS < 46:
    # Her blok aracı yeniden başlatır ve adres listesi her seferinde baştan yüklenir (milyonlarca adreste
    # dakikalar). 2^46'lık bir blok ~10 Gkey/s'de ~2 saat sürer, yükleme süresi bunun yanında küçük kalır.
    BLOCK_BITS = 46
    print('ℹ️  multi + blocks: liste her blokta yeniden yüklendiği için BLOCK_BITS = 46 yapıldı.')
if BLOCK_BITS > RANGE_BITS - 2:          # test için küçük aralık seçildiyse bloğu aralığa sığdır
    BLOCK_BITS = RANGE_BITS - 2
    print(f'ℹ️  Küçük aralık: BLOCK_BITS = {BLOCK_BITS} olarak ayarlandı.')
if MULTI:
    print(f'Mod   : multi (çoklu adres) · liste={ADDRESS_LIST}' + (f' · dosya={ADDRESS_FILE}' if ADDRESS_FILE else ''))
else:
    print(f'Mod   : puzzle · Hedef : {TARGET_ADDRESS}')
print(f'Aralık: {RANGE_START:x} : {RANGE_END:x}  (2^{RANGE_BITS} = {RANGE_SIZE:.3e} anahtar)')

In [ ]:
#@title 🧰 2) Yardımcı fonksiyonlar (secp256k1 doğrulama, süreç yönetimi, çıktı ayrıştırma)
import os, re, sys, json, time, math, glob, queue, shutil, signal, secrets, hashlib, platform, tempfile, threading, subprocess, statistics, datetime

assert sys.version_info >= (3, 8), 'Python 3.8 veya üstü gerekli.'
IS_WIN   = os.name == 'nt'
IN_COLAB = 'google.colab' in sys.modules or 'COLAB_RELEASE_TAG' in os.environ
if not IS_WIN:
    import pty
# Yerel çalıştırmada araçlar ve kayıtlar repo/çalışma klasörüne değil kullanıcı klasörüne yazılır
# (bulunan anahtar yanlışlıkla bir git commit'ine girmesin). Ortam değişkeniyle değiştirilebilir.
HOME_DIR = '/content' if IN_COLAB else os.path.normpath(os.environ.get('PUZZLE_HOME') or
                                                          os.path.join(os.path.expanduser('~'), 'vanitysearch-puzzle'))

# ---------------------------------------------------------------------------
# Saf Python secp256k1 + adres üretimi. GPU aracı bir anahtar bulduğunda onu
# buradan BAĞIMSIZ olarak doğruluyoruz; sahte/yanlış pozitif asla "bulundu" sayılmaz.
# ---------------------------------------------------------------------------
_P = 2**256 - 2**32 - 977
_G = (0x79BE667EF9DCBBAC55A06295CE870B07029BFCDB2DCE28D959F2815B16F81798,
      0x483ADA7726A3C4655DA4FBFC0E1108A8FD17B448A68554199C47D08FFB10D4B8)

def _ec_add(a, b):
    if a is None: return b
    if b is None: return a
    if a[0] == b[0]:
        if (a[1] + b[1]) % _P == 0: return None
        l = 3 * a[0] * a[0] * pow(2 * a[1], -1, _P) % _P
    else:
        l = (b[1] - a[1]) * pow(b[0] - a[0], -1, _P) % _P
    x = (l * l - a[0] - b[0]) % _P
    return (x, (l * (a[0] - x) - a[1]) % _P)

def pubkey_compressed(k):
    r, q = None, _G
    while k:
        if k & 1: r = _ec_add(r, q)
        q = _ec_add(q, q); k >>= 1
    return bytes([2 + (r[1] & 1)]) + r[0].to_bytes(32, 'big')

def _rmd160_py(msg):
    # OpenSSL 3 bazı sistemlerde ripemd160'ı kapatıyor; saf Python yedek.
    rol = lambda x, n: ((x << n) | (x >> (32 - n))) & 0xFFFFFFFF
    F = [lambda x, y, z: x ^ y ^ z,
         lambda x, y, z: (x & y) | (~x & z),
         lambda x, y, z: (x | ~y) ^ z,
         lambda x, y, z: (x & z) | (y & ~z),
         lambda x, y, z: x ^ (y | ~z)]
    KL = [0x00000000, 0x5A827999, 0x6ED9EBA1, 0x8F1BBCDC, 0xA953FD4E]
    KR = [0x50A28BE6, 0x5C4DD124, 0x6D703EF3, 0x7A6D76E9, 0x00000000]
    RL = [0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15, 7,4,13,1,10,6,15,3,12,0,9,5,2,14,11,8,
          3,10,14,4,9,15,8,1,2,7,0,6,13,11,5,12, 1,9,11,10,0,8,12,4,13,3,7,15,14,5,6,2,
          4,0,5,9,7,12,2,10,14,1,3,8,11,6,15,13]
    RR = [5,14,7,0,9,2,11,4,13,6,15,8,1,10,3,12, 6,11,3,7,0,13,5,10,14,15,8,12,4,9,1,2,
          15,5,1,3,7,14,6,9,11,8,12,2,10,0,4,13, 8,6,4,1,3,11,15,0,5,12,2,13,9,7,10,14,
          12,15,10,4,1,5,8,7,6,2,13,14,0,3,9,11]
    SL = [11,14,15,12,5,8,7,9,11,13,14,15,6,7,9,8, 7,6,8,13,11,9,7,15,7,12,15,9,11,7,13,12,
          11,13,6,7,14,9,13,15,14,8,13,6,5,12,7,5, 11,12,14,15,14,15,9,8,9,14,5,6,8,6,5,12,
          9,15,5,11,6,8,13,12,5,12,13,14,11,8,5,6]
    SR = [8,9,9,11,13,15,15,5,7,7,8,11,14,14,12,6, 9,13,15,7,12,8,9,11,7,7,12,7,6,15,13,11,
          9,7,15,11,8,6,6,14,12,13,5,14,13,13,7,5, 15,5,8,11,14,14,6,14,6,9,12,9,12,5,15,8,
          8,5,12,9,12,5,14,6,8,13,6,5,15,13,11,11]
    h = [0x67452301, 0xEFCDAB89, 0x98BADCFE, 0x10325476, 0xC3D2E1F0]
    msg = bytes(msg)
    ml = len(msg) * 8
    msg += b'\x80' + b'\x00' * ((55 - len(msg)) % 64) + ml.to_bytes(8, 'little')
    for off in range(0, len(msg), 64):
        X = [int.from_bytes(msg[off + 4 * i: off + 4 * i + 4], 'little') for i in range(16)]
        al, bl, cl, dl, el = h
        ar, br, cr, dr, er = h
        for j in range(80):
            r = j // 16
            t = (rol((al + F[r](bl, cl, dl) + X[RL[j]] + KL[r]) & 0xFFFFFFFF, SL[j]) + el) & 0xFFFFFFFF
            al, el, dl, cl, bl = el, dl, rol(cl, 10), bl, t
            t = (rol((ar + F[4 - r](br, cr, dr) + X[RR[j]] + KR[r]) & 0xFFFFFFFF, SR[j]) + er) & 0xFFFFFFFF
            ar, er, dr, cr, br = er, dr, rol(cr, 10), br, t
        t = (h[1] + cl + dr) & 0xFFFFFFFF
        h[1] = (h[2] + dl + er) & 0xFFFFFFFF
        h[2] = (h[3] + el + ar) & 0xFFFFFFFF
        h[3] = (h[4] + al + br) & 0xFFFFFFFF
        h[4] = (h[0] + bl + cr) & 0xFFFFFFFF
        h[0] = t
    return b''.join(x.to_bytes(4, 'little') for x in h)

def ripemd160(b):
    try:
        return hashlib.new('ripemd160', b).digest()
    except Exception:
        return _rmd160_py(b)

_sha256d = lambda b: hashlib.sha256(hashlib.sha256(b).digest()).digest()
_B58 = '123456789ABCDEFGHJKLMNPQRSTUVWXYZabcdefghijkmnopqrstuvwxyz'

def b58check_encode(payload):
    b = payload + _sha256d(payload)[:4]
    n, s = int.from_bytes(b, 'big'), ''
    while n:
        n, r = divmod(n, 58); s = _B58[r] + s
    return '1' * (len(b) - len(b.lstrip(b'\0'))) + s

def b58check_decode(s):
    n = 0
    for c in s: n = n * 58 + _B58.index(c)
    raw = b'\0' * (len(s) - len(s.lstrip('1'))) + n.to_bytes((n.bit_length() + 7) // 8, 'big')
    data, chk = raw[:-4], raw[-4:]
    if _sha256d(data)[:4] != chk: raise ValueError('base58 checksum hatalı')
    return data

_BECH = 'qpzry9x8gf2tvdw0s3jn54khce6mua7l'

def _bech32_polymod(values):
    chk = 1
    for v in values:
        b = chk >> 25
        chk = (chk & 0x1ffffff) << 5 ^ v
        for i, g in enumerate((0x3b6a57b2, 0x26508e6d, 0x1ea119fa, 0x3d4233dd, 0x2a1462b3)):
            chk ^= g if (b >> i) & 1 else 0
    return chk

def p2wpkh_encode(h160):
    """bc1q... adresi (BIP-173, tanık sürümü 0)."""
    acc, bits, data = 0, 0, [0]
    for byte in h160:
        acc = acc << 8 | byte; bits += 8
        while bits >= 5:
            bits -= 5; data.append(acc >> bits & 31)
    if bits: data.append(acc << (5 - bits) & 31)
    hrp = [ord(c) >> 5 for c in 'bc'] + [0] + [ord(c) & 31 for c in 'bc']
    pm = _bech32_polymod(hrp + data + [0] * 6) ^ 1
    return 'bc1' + ''.join(_BECH[d] for d in data + [pm >> 5 * (5 - i) & 31 for i in range(6)])

def hash160_from_key(k): return ripemd160(hashlib.sha256(pubkey_compressed(k)).digest())
def addr_from_key(k):  return b58check_encode(b'\x00' + hash160_from_key(k))
def bech32_from_key(k): return p2wpkh_encode(hash160_from_key(k))
def wif_from_key(k):   return b58check_encode(b'\x80' + k.to_bytes(32, 'big') + b'\x01')
def key_from_wif(w):   return int.from_bytes(b58check_decode(w)[1:33], 'big')

# Bilinen test vektörleri (özel anahtar = 1)
assert _rmd160_py(b'abc').hex() == '8eb208f7e05d987a9b044a8e98c6b087f15a0bfc'
assert addr_from_key(1) == '1BgGZ9tcN4rm9KBzDn7KprQz87SZ26SAMH'
assert wif_from_key(1) == 'KwDiBf89QgGbjEhKnhXJuH7LrciVrZi3qYjgd9M7rFU73sVHnoWn'
assert bech32_from_key(1) == 'bc1qw508d6qejxtdg4y5r3zarvary0c5xw7kv8f3t4'

# ---------------------------------------------------------------------------
# Kabuk ve süreç yönetimi
# ---------------------------------------------------------------------------
def sh(cmd, check=True, quiet=False, tail=3000):
    """cmd metinse kabukta (Linux: bash, Windows: cmd), listeyse doğrudan çalışır."""
    kw = dict(shell=True, executable=None if IS_WIN else '/bin/bash') if isinstance(cmd, str) else {}
    try:
        r = subprocess.run(cmd, text=True, encoding='utf-8', errors='replace', capture_output=True,
                           stdin=subprocess.DEVNULL, **kw)
    except FileNotFoundError as e:                 # liste halindeki komutun ikilisi yok
        r = subprocess.CompletedProcess(cmd, 127, '', str(e))
    if not quiet or (check and r.returncode):
        out = (r.stdout + r.stderr).strip()
        if out: print(out[-tail:])
    if check and r.returncode:
        raise RuntimeError(f'Komut başarısız (rc={r.returncode}): {cmd}')
    return r.stdout

def _stop_proc(p):
    if p.poll() is not None: return
    if IS_WIN:
        steps = ((lambda: p.send_signal(signal.CTRL_BREAK_EVENT), 5), (p.kill, 5))
    else:
        steps = ((lambda: os.killpg(p.pid, signal.SIGINT), 5), (lambda: os.killpg(p.pid, signal.SIGKILL), 5))
    for act, wait in steps:
        try: act()
        except (ProcessLookupError, OSError): pass
        try: p.wait(wait); return
        except subprocess.TimeoutExpired: pass

def stream(cmd, cwd=None, timeout=None, info=None):
    """Aracı çalıştırıp çıktısını satır satır verir. Linux/macOS'ta sahte terminal (pty) altında çalışır,
    böylece çıktı tamponlanmaz; Windows'ta boru (pipe) kullanılır (araç her satırda fflush yapıyor).
    Her satırı (\\r ve \\n ile bölünmüş) yield eder; çıktı yokken ~1 sn'de bir '' (nabız) yield eder.
    Döngüden çıkılınca (break / hata / Ctrl+C) süreç (grubu) öldürülür."""
    if IS_WIN:
        p = subprocess.Popen(cmd, cwd=cwd, stdin=subprocess.DEVNULL, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                             creationflags=subprocess.CREATE_NEW_PROCESS_GROUP)
        rfd, close_fd = p.stdout.fileno(), p.stdout.close
    else:
        master, slave = pty.openpty()
        p = subprocess.Popen(cmd, cwd=cwd, stdin=subprocess.DEVNULL, stdout=slave, stderr=slave,
                             start_new_session=True, close_fds=True)
        os.close(slave)
        rfd, close_fd = master, (lambda: os.close(master))
    chunks = queue.Queue()
    def reader():
        while True:
            try: data = os.read(rfd, 65536)
            except OSError: data = b''               # pty: süreç bitince EIO
            chunks.put(data)
            if not data: return
    th = threading.Thread(target=reader, daemon=True); th.start()
    buf, t_end = b'', (time.time() + timeout) if timeout else None
    try:
        while True:
            if t_end and time.time() > t_end: break
            try: data = chunks.get(timeout=1.0)
            except queue.Empty:
                yield ''
                continue
            if not data: break
            buf += data
            *lines, buf = re.split(rb'[\r\n]+', buf)
            for ln in lines:
                s = ln.decode('utf-8', 'replace').strip()
                if s: yield s
        if buf.strip(): yield buf.decode('utf-8', 'replace').strip()
    finally:
        _stop_proc(p)
        th.join(5)
        try: close_fd()
        except OSError: pass
        if info is not None: info['rc'] = p.returncode

# ---------------------------------------------------------------------------
# VanitySearch çıktısını ayrıştırma
#   "8812.3 MK/s - 1234 BKeys - 2^40.1 [..] ..."   /  "Priv (HEX): 0x...."
#   "  CandKey: ..."  (aracın kendi CPU kontrolü başarısız olsa bile yazdırdığı aday anahtar)
# WIF satırları bilerek ayrıştırılmıyor: VanitySearch'ün CPU SHA-256'sı bazı derleyicilerle hatalı checksum üretiyor.
# ---------------------------------------------------------------------------
_N = 0xFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFEBAAEDCE6AF48A03BBFD25E8CD0364141
_RX = {
    'vs_hex':   re.compile(r'Priv \(HEX\):\s*0x\s*([0-9A-Fa-f]+)'),
    'vs_cand':  re.compile(r'CandKey:\s*([0-9A-Fa-f]+)'),
    'vs_speed': re.compile(r'([\d.]+)\s*MK/s\s*-\s*([\d.]+)\s*BKeys'),
}

def parse_line(line):
    """('keys', [aday anahtarlar]) | ('speed', Mkeys/s, bu süreçte taranan toplam anahtar) | None"""
    m = _RX['vs_hex'].search(line)
    if m: return ('keys', [int(m.group(1), 16)])
    m = _RX['vs_cand'].search(line)
    if m:
        c = int(m.group(1), 16) % _N           # araç negatifini de denemiş olabilir: ikisini de aday say
        return ('keys', [c, (_N - c) % _N])
    m = _RX['vs_speed'].search(line)
    if m: return ('speed', float(m.group(1)), int(float(m.group(2)) * 1e9))
    return None

def fmt_num(x):
    for unit, d in (('E', 1e18), ('P', 1e15), ('T', 1e12), ('G', 1e9), ('M', 1e6), ('K', 1e3)):
        if abs(x) >= d: return f'{x / d:.2f} {unit}'
    return f'{x:.0f}'

def fmt_prob(p):
    """Olasılık / oran: büyükse yüzde, küçükse '1 / N'."""
    if p <= 0: return '0'
    if p >= 1e-3: return f'%{min(p, 1.0) * 100:.2f}'
    return f'≈ 1 / {1 / p:,.0f}'

def fmt_dur(s):
    s = int(s)
    d, s = divmod(s, 86400); h, s = divmod(s, 3600); m, s = divmod(s, 60)
    return (f'{d}g ' if d else '') + f'{h:02d}:{m:02d}:{s:02d}'

print('✅ Yardımcılar yüklendi, secp256k1/RIPEMD160/Base58/Bech32 test vektörleri geçti.')

In [ ]:
#@title 🖥️ 3) GPU, hedef/adres listesi, bakiye ve kayıt klasörü kontrolü
import urllib.request
try:
    from IPython.display import display, HTML, Javascript
except ImportError:                                  # Jupyter dışı (düz python) çalıştırma
    HTML = Javascript = lambda x: x
    class _Disp:
        def update(self, x): pass
    def display(x, display_id=False):
        if isinstance(x, str) and x.startswith('<div'): print(re.sub('<[^>]+>', ' ', x))
        return _Disp()

FOUND_KEY = None      # puzzle modunda bulunan anahtar
FOUND_KEYS = []       # multi modunda bulunan anahtarlar

# --- GPU ---
# nvidia-smi GPU'ları PCI sırasıyla numaralar; CUDA'yı (dolayısıyla -gpuId'yi) aynı sıraya zorla.
os.environ['CUDA_DEVICE_ORDER'] = 'PCI_BUS_ID'

def gpu_info_cuda(idx):
    """nvidia-smi yoksa ya da compute_cap alanını bilmiyorsa (eski sürücü): CUDA sürücü API'si (ctypes)."""
    import ctypes
    lib = ctypes.WinDLL('nvcuda.dll') if IS_WIN else ctypes.CDLL('libcuda.so.1')
    def ok(rc, what):
        if rc: raise RuntimeError(f'{what} hata kodu {rc}')
    ok(lib.cuInit(0), 'cuInit')
    dev, n, mj, mn, ver = ctypes.c_int(), ctypes.c_int(), ctypes.c_int(), ctypes.c_int(), ctypes.c_int()
    ok(lib.cuDeviceGetCount(ctypes.byref(n)), 'cuDeviceGetCount')
    if idx >= n.value: raise RuntimeError(f'GPU_ID={idx} ama {n.value} GPU var')
    ok(lib.cuDeviceGet(ctypes.byref(dev), idx), 'cuDeviceGet')
    name, mem = ctypes.create_string_buffer(256), ctypes.c_size_t()
    lib.cuDeviceGetName(name, 256, dev)
    lib.cuDeviceGetAttribute(ctypes.byref(mj), 75, dev)       # COMPUTE_CAPABILITY_MAJOR
    lib.cuDeviceGetAttribute(ctypes.byref(mn), 76, dev)       # COMPUTE_CAPABILITY_MINOR
    lib.cuDeviceTotalMem_v2(ctypes.byref(mem), dev)
    lib.cuDriverGetVersion(ctypes.byref(ver))
    return (name.value.decode(), f'{mj.value}.{mn.value}', f'{mem.value >> 20} MiB',
            f'CUDA {ver.value // 1000}.{ver.value % 1000 // 10}')

def driver_cuda_version():
    """Sürücünün desteklediği en yüksek CUDA sürümü, ör. (12, 8); bilinmiyorsa None."""
    m = re.search(r'CUDA Version:\s*(\d+)\.(\d+)', sh(['nvidia-smi'], check=False, quiet=True))
    return (int(m.group(1)), int(m.group(2))) if m else None

try:
    q = sh(['nvidia-smi', '-i', str(GPU_ID), '--query-gpu=name,compute_cap,memory.total,driver_version',
            '--format=csv,noheader'], quiet=True)
    GPU_NAME, GPU_CC, GPU_MEM, GPU_DRIVER = [x.strip() for x in q.strip().splitlines()[0].split(',')]
    CC = int(GPU_CC.replace('.', ''))
except Exception as e:
    print('ℹ️  nvidia-smi ile okunamadı, CUDA sürücü API\'si deneniyor:', str(e)[:200])
    try:
        GPU_NAME, GPU_CC, GPU_MEM, GPU_DRIVER = gpu_info_cuda(GPU_ID)
        CC = int(GPU_CC.replace('.', ''))
    except Exception as e2:
        raise RuntimeError('❌ NVIDIA GPU bulunamadı. Bu notebook NVIDIA GPU + sürücü ister '
                           '(Colab: Çalışma zamanı türünü GPU yap; Docker: --gpus all ile başlat).') from e2
print(f'GPU    : {GPU_NAME}  (GPU_ID {GPU_ID}, compute {GPU_CC}, {GPU_MEM}, driver {GPU_DRIVER})')
print(f'Ortam  : {"Google Colab" if IN_COLAB else platform.system() + " " + platform.machine()} · Python {platform.python_version()}')
if platform.machine().lower() not in ('x86_64', 'amd64'):
    raise RuntimeError(f'❌ {platform.machine()} işlemci desteklenmiyor: VanitySearch\'ün CPU tarafı x86-64 SSE kodu içeriyor.')
if CC < 120:
    print('⚠️  Bu bir Blackwell GPU değil. Notebook yine çalışır ama hız beklenenden düşük olur.')

# --- Hedef / adres listesi ---
TARGET_H160 = None
ADDRESS_CACHE = set()           # multi: dosyada bulunduğu doğrulanan adresler (tekrar taramamak için)
MULTI_BECH32 = (ADDRESS_LIST == 'bech32')   # multi: listenin adres türü (custom için dosyadan okunur)

def list_addr_from_key(k):
    """multi modunda bir anahtarın, listedeki türden (1... ya da bc1q...) adresi."""
    return bech32_from_key(k) if MULTI_BECH32 else addr_from_key(k)

if not MULTI:
    # puzzle modu: tek adres + bakiye kontrolü
    _payload = b58check_decode(TARGET_ADDRESS)
    assert len(_payload) == 21 and _payload[0] == 0, 'Hedef bir P2PKH (1...) adresi olmalı'
    TARGET_H160 = _payload[1:].hex()
    print(f'Hedef  : {TARGET_ADDRESS}  hash160={TARGET_H160}')

    def address_info(addr):
        last = None
        for base in ('https://mempool.space/api', 'https://blockstream.info/api'):
            try:
                req = urllib.request.Request(f'{base}/address/{addr}', headers={'User-Agent': 'puzzle-notebook'})
                with urllib.request.urlopen(req, timeout=20) as r:
                    d = json.load(r)
                cs = d['chain_stats']
                return (cs['funded_txo_sum'] - cs['spent_txo_sum']) / 1e8, cs['spent_txo_count']
            except Exception as e:
                last = e
        print('⚠️  Bakiye sorgulanamadı:', last)
        return None, None

    BALANCE, SPENT_COUNT = address_info(TARGET_ADDRESS)
    if BALANCE is not None:
        print(f'Bakiye : {BALANCE:.8f} BTC')
        if BALANCE == 0 and PUZZLE_BITS > 50:
            raise RuntimeError('❌ Adreste bakiye yok: bulmaca zaten çözülmüş. Çalıştırmayı durdurdum.')
        if BALANCE == 0:
            print('ℹ️  Test: bu bulmaca zaten çözülmüş, sadece deneme amaçlı çalıştırılıyor.')
        if SPENT_COUNT:
            print('⚠️  Bu adresten harcama yapılmış → public key açıkta. Bu durumda brute-force yerine Kangaroo/BSGS çok daha hızlıdır.')
else:
    # multi modu: adres listesi dosyası (indiriciden ya da hazır dosya)
    if not ADDRESS_FILE:
        ADDRESS_FILE = os.path.join(HOME_DIR, f'rich_{ADDRESS_LIST}.txt')   # büyük dosya: Drive'a değil yerel diske
    print(f'Adres listesi: {ADDRESS_FILE}')
    print('ℹ️  Çoklu adres modu: rastgele bir aralıkta fonlu cüzdan araması. Normal cüzdanların anahtarları '
          '2^256 üzerinde düzgün dağıldığı için gerçekçi bulma şansı YOKTUR; bu daha çok filtreyi ölçeklemek '
          've kendi zayıf anahtarlarını denemek içindir.')

    def _detect_list_type():
        """Listenin ilk adresine göre türü belirler (araç da ilk adresin türünü kullanır)."""
        global MULTI_BECH32
        with open(ADDRESS_FILE, encoding='utf-8', errors='replace') as f:
            for line in f:
                line = line.strip()
                if line and not line.startswith('#'):
                    MULTI_BECH32 = line.lower().startswith('bc1')
                    break
        print(f'Adres türü: {"BECH32 (bc1q...)" if MULTI_BECH32 else "P2PKH (1...)"}')

    def ensure_address_file():
        """ADDRESS_FILE yoksa tools/fetch_addresses.py ile Hugging Face'den indir (gated: jeton gerekir)."""
        global ADDRESS_FILE
        if not ADDRESS_FILE:                       # hücre sırası ne olursa olsun yolu kendin çöz
            ADDRESS_FILE = os.path.join(HOME_DIR, f'rich_{ADDRESS_LIST}.txt')
        ADDRESS_FILE = os.path.abspath(ADDRESS_FILE)   # göreli yol indiricinin CWD'sine düşmesin
        if os.path.exists(ADDRESS_FILE) and os.path.getsize(ADDRESS_FILE) > 0:
            print(f'Adres listesi hazır: {ADDRESS_FILE} ({os.path.getsize(ADDRESS_FILE)} bayt)')
            _detect_list_type()
            return
        if ADDRESS_LIST == 'custom':
            raise RuntimeError(f'ADDRESS_FILE bulunamadı: {ADDRESS_FILE} (custom modunda dosyayı kendin ver)')
        script = os.path.join(VS_DIR, 'tools', 'fetch_addresses.py')
        tok = HF_TOKEN or os.environ.get('HF_TOKEN', '') or os.environ.get('HUGGING_FACE_HUB_TOKEN', '')
        cmd = [sys.executable, script, ADDRESS_LIST, '-o', ADDRESS_FILE] + (['--token', tok] if tok else [])
        print('Adres listesi indiriliyor (yüzlerce MB olabilir, birkaç dakika sürebilir)...')
        sh(cmd, tail=4000)
        if not (os.path.exists(ADDRESS_FILE) and os.path.getsize(ADDRESS_FILE) > 0):
            raise RuntimeError('Adres listesi indirilemedi. Dataset gated: ayarlardaki HF_TOKEN\'i doldur ve '
                               'dataset sayfasında erişimi kabul et.')
        _detect_list_type()

    def address_in_file(addr):
        """addr listedeyse True. Büyük dosyayı RAM'e almamak için akış halinde tarar (yalnızca bir eşleşme gelince çağrılır)."""
        try:
            with open(ADDRESS_FILE, encoding='utf-8', errors='replace') as f:
                for line in f:
                    if line.strip() == addr:
                        return True
        except FileNotFoundError:
            pass
        return False

# --- Kayıt klasörü (Drive varsa oraya: oturum koparsa bulunan anahtar kaybolmasın) ---
_tag = f'multi_{ADDRESS_LIST}' if MULTI else f'puzzle{PUZZLE_BITS}'
STATE_DIR = os.path.join(HOME_DIR, f'btc_{_tag}')
if USE_DRIVE and IN_COLAB:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        STATE_DIR = f'/content/drive/MyDrive/btc_{_tag}'
    except Exception as e:
        print('⚠️  Google Drive bağlanamadı, yerel klasör kullanılacak:', e)
os.makedirs(STATE_DIR, exist_ok=True)
STATS_PATH  = os.path.join(STATE_DIR, 'stats.json')
BLOCKS_PATH = os.path.join(STATE_DIR, f'blocks_done_b{BLOCK_BITS}.txt')
LOG_PATH    = os.path.join(STATE_DIR, 'log.txt')
FOUND_PATHS = sorted({os.path.join(STATE_DIR, 'FOUND_KEY.txt'), os.path.join(HOME_DIR, f'FOUND_KEY_{_tag}.txt')})
VS_OUT      = os.path.join(STATE_DIR, 'vanitysearch_found.txt')
print(f'Kayıt  : {STATE_DIR}')

def log(msg):
    line = f'[{datetime.datetime.now(datetime.timezone.utc):%Y-%m-%d %H:%M:%S} UTC] {msg}'
    print(line)
    with open(LOG_PATH, 'a', encoding='utf-8') as f: f.write(line + '\n')

def load_stats():
    try:
        with open(STATS_PATH, encoding='utf-8') as f: return json.load(f)
    except Exception:
        return {'total_keys': 0, 'seconds': 0.0, 'sessions': 0}

def save_stats(st):
    tmp = STATS_PATH + '.tmp'
    with open(tmp, 'w', encoding='utf-8') as f: json.dump(st, f, indent=1)
    os.replace(tmp, STATS_PATH)

def load_done_blocks():
    try:
        with open(BLOCKS_PATH) as f: return {int(x, 16) for x in f.read().split()}
    except FileNotFoundError:
        return set()

def mark_block_done(idx):
    with open(BLOCKS_PATH, 'a') as f: f.write(f'{idx:x}\n')

def _record_found(k, addr, source):
    """Doğrulanmış bir anahtarı ekrana basar ve FOUND dosyalarına yazar."""
    wif = wif_from_key(k)
    head = f'PUZZLE {PUZZLE_BITS} COZULDU' if not MULTI else 'ADRES BULUNDU'
    txt = (f'{head}  ({datetime.datetime.now(datetime.timezone.utc):%Y-%m-%d %H:%M:%S} UTC, {source})\n'
           f'Address          : {addr}\n'
           f'Private key (HEX): {k:064x}\n'
           f'WIF (compressed) : {wif}\n' +
           (f'Electrum için    : p2wpkh:{wif}  (bc1q adresi için anahtar türü öneki)\n' if addr.startswith('bc1') else '') +
           f'Public key       : {pubkey_compressed(k).hex()}\n')
    for path in FOUND_PATHS:
        try:
            with open(path, 'a', encoding='utf-8') as f:
                f.write(txt + '\n'); f.flush(); os.fsync(f.fileno())
        except Exception as e:
            print('yazılamadı', path, e)
    log('🎉🎉🎉 ANAHTAR BULUNDU VE DOĞRULANDI! 🎉🎉🎉')
    print('=' * 78 + '\n' + txt + '=' * 78)
    print('Kaydedildi:', ', '.join(FOUND_PATHS))
    display(HTML(f'<div style="font-size:22px;padding:16px;border:4px solid #1a7f37;border-radius:8px">'
                 f'🎉 <b>BULUNDU!</b><br>{addr}<br><code>{k:064x}</code><br>WIF: <code>{wif}</code><br>'
                 f'<b>Fonları açık mempool üzerinden gönderme</b>, en alttaki talimatları oku.</div>'))
    display(Javascript('''(async()=>{const c=new AudioContext();for(let i=0;i<20;i++){const o=c.createOscillator();
        o.frequency.value=880;o.connect(c.destination);o.start(c.currentTime+i*0.5);o.stop(c.currentTime+i*0.5+0.25);}})()'''))

def on_found(cands, source=''):
    """GPU aracının bildirdiği aday anahtar(lar)ı BAĞIMSIZ doğrular; doğruysa kaydeder.
    puzzle modunda True döner (arama biter); multi modunda False döner (aramaya devam)."""
    global FOUND_KEY
    if MULTI:
        for c in cands:
            if not (RANGE_START <= c <= RANGE_END) or c in FOUND_KEYS:
                continue
            a = list_addr_from_key(c)
            if a in ADDRESS_CACHE or address_in_file(a):
                ADDRESS_CACHE.add(a)
                FOUND_KEYS.append(c)
                _record_found(c, a, source)
            else:
                log(f'⚠️  {source}: {c:x} -> {a} listede DEĞİL (yok sayıldı).')
        return False
    if FOUND_KEY is not None and FOUND_KEY in cands:
        return True                                  # zaten kaydedildi
    k = next((c for c in cands if RANGE_START <= c <= RANGE_END and addr_from_key(c) == TARGET_ADDRESS), None)
    if k is None:
        log(f"⚠️  {source}: bildirilen anahtar(lar) {', '.join(f'{c:x}' for c in cands)} hedefle EŞLEŞMEDİ (yok sayıldı).")
        return False
    FOUND_KEY = k
    _record_found(k, TARGET_ADDRESS, source)
    return True

# Daha önceki bir oturumda bulunmuş mu? Bulunanlar yüklenir ki aynı anahtar her oturumda yeniden
# "BULUNDU" diye kaydedilmesin (arama hücresi -o dosyasını her başlangıçta yeniden tarıyor).
for p in FOUND_PATHS:
    if os.path.exists(p):
        _txt = open(p, encoding='utf-8', errors='replace').read()
        print(f'‼️  {p} zaten var, içeriğine bak:\n' + _txt)
        for _k in re.findall(r'Private key \(HEX\):\s*([0-9a-fA-F]+)', _txt):
            _k = int(_k, 16)
            if MULTI and _k not in FOUND_KEYS:
                FOUND_KEYS.append(_k)
            elif not MULTI and addr_from_key(_k) == TARGET_ADDRESS:
                FOUND_KEY = _k
if not MULTI and FOUND_KEY is not None:
    print(f'‼️  Bu bulmacanın anahtarı daha önce bulunmuş: {FOUND_KEY:064x} (arama hücresi aramayı başlatmaz)')

_st = load_stats()
print(f'Önceki oturumlar: {_st["sessions"]} oturum, {fmt_num(_st["total_keys"])} anahtar, {fmt_dur(_st["seconds"])}')

In [ ]:
#@title 🔧 4) CUDA derleyicisi + VanitySearch-Blackwell derleme (GPU'ya özel, ilk seferde ~2-5 dk)
WORK = os.path.join(HOME_DIR, 'tools')
os.makedirs(WORK, exist_ok=True)

# Linux paket kurulumu: root değilsek parolasız sudo gerekir; apt yoksa (Fedora, Arch, ...) paketler elle kurulur.
APT  = None if IS_WIN else shutil.which('apt-get')
SUDO = '' if IS_WIN or os.geteuid() == 0 else ('sudo -n ' if shutil.which('sudo') else None)
CAN_APT = bool(APT) and SUDO is not None and (SUDO == '' or subprocess.run(['sudo', '-n', 'true'],
                                              capture_output=True).returncode == 0)

def pkg_install(pkgs):
    if not CAN_APT:
        raise RuntimeError(f'Şu paket(ler)i elle kur: {pkgs}  (apt-get yok ya da root/parolasız sudo yetkisi yok).')
    sh(f'{SUDO}apt-get update -qq', check=False, quiet=True)
    sh(f'{SUDO}env DEBIAN_FRONTEND=noninteractive apt-get install -y -qq {pkgs}', quiet=True)

_missing = [t for t in (['git'] if IS_WIN else ['git', 'make', 'g++']) if not shutil.which(t)]
if _missing and IS_WIN:
    raise RuntimeError('git bulunamadı: https://git-scm.com/download/win adresinden kur, sonra Jupyter\'i yeniden başlat.')
if _missing:
    pkg_install(' '.join(_missing))

def nvcc_archs(nvcc):
    try:
        out = subprocess.run([nvcc, '--list-gpu-arch'], capture_output=True, text=True, timeout=60).stdout
    except Exception:
        return set()
    return {int(x) for x in re.findall(r'compute_(\d+)', out)}

def _cuda_ver_key(path):
    m = re.search(r'(\d+)\.(\d+)', path)
    return (int(m.group(1)), int(m.group(2))) if m else (0, 0)

def nvcc_candidates():
    if IS_WIN:
        roots = [os.environ.get('CUDA_PATH')] + sorted(
            glob.glob(os.path.expandvars(r'%ProgramFiles%\NVIDIA GPU Computing Toolkit\CUDA\v*')), key=_cuda_ver_key, reverse=True)
        return [shutil.which('nvcc')] + [os.path.join(r, 'bin', 'nvcc.exe') for r in roots if r]
    return ([shutil.which('nvcc'), '/usr/local/cuda/bin/nvcc'] +
            sorted(glob.glob('/usr/local/cuda-*/bin/nvcc'), key=_cuda_ver_key, reverse=True) +
            ([os.path.join(os.environ['CONDA_PREFIX'], 'bin', 'nvcc')] if os.environ.get('CONDA_PREFIX') else []))

def pick_nvcc():
    seen, best = set(), (None, set())
    for n in nvcc_candidates():
        if not n or not os.path.exists(n) or os.path.realpath(n) in seen: continue
        seen.add(os.path.realpath(n))
        a = nvcc_archs(n)
        ver = re.search(r'release ([\d.]+)', sh([n, '--version'], check=False, quiet=True))
        print(f'  nvcc {n}  (CUDA {ver.group(1) if ver else "?"}, en yüksek arch: {max(a) if a else "?"})')
        if CC in a: return n, a
        if a and (not best[1] or max(a) > max(best[1])): best = (n, a)
    return best

def install_cuda_nvcc():
    """NVIDIA'nın resmi apt deposundan sadece derleyici + runtime başlıkları (tam toolkit değil).
    Sürüm: bu GPU'yu destekleyen ve sürücünün çalıştırabildiği (nvidia-smi 'CUDA Version') ilk sürüm."""
    if not CAN_APT:
        raise RuntimeError('otomatik kurulum yalnızca apt + root/parolasız sudo ile çalışır')
    drv = driver_cuda_version() or (99, 0)
    need = (12, 9) if CC >= 121 else (12, 8) if CC >= 100 else (11, 8) if CC >= 89 else (11, 0)
    prefs = [(12, 8), (12, 9), (13, 0), (12, 6), (12, 4), (12, 2), (11, 8)]
    vers = [v for v in prefs if need <= v <= drv and not (v >= (13, 0) and CC < 75)]
    if not vers:
        raise RuntimeError(f'sürücü CUDA {drv[0]}.{drv[1]} destekliyor, sm_{CC} için en az CUDA {need[0]}.{need[1]} gerekli: sürücüyü güncelle')
    def apt_has(v):
        pol = sh(f'apt-cache policy cuda-nvcc-{v[0]}-{v[1]}', check=False, quiet=True)
        return bool(pol.strip()) and 'Candidate: (none)' not in pol
    sh(f'{SUDO}apt-get update -qq', check=False, quiet=True)
    if not any(apt_has(v) for v in vers):
        osid = sh('. /etc/os-release && echo ${ID}${VERSION_ID//./}', quiet=True).strip()
        deb = os.path.join(tempfile.gettempdir(), 'cuda-keyring.deb')
        urllib.request.urlretrieve(f'https://developer.download.nvidia.com/compute/cuda/repos/{osid}/x86_64/'
                                   f'cuda-keyring_1.1-1_all.deb', deb)
        sh(f'{SUDO}dpkg -i {deb} && {SUDO}apt-get update -qq', quiet=True)
    for v in vers:
        if apt_has(v):
            print(f'  → CUDA {v[0]}.{v[1]} nvcc kuruluyor...')
            pkg_install(f'cuda-nvcc-{v[0]}-{v[1]} cuda-cudart-dev-{v[0]}-{v[1]}')
            return
    raise RuntimeError(f'apt deposunda uygun cuda-nvcc paketi yok (denenen: {vers})')

print('CUDA derleyicileri:')
NVCC, ARCHS = pick_nvcc()
if CC not in ARCHS and AUTO_INSTALL_CUDA and not IS_WIN:
    print(f'→ Mevcut nvcc sm_{CC} için yerel kod üretemiyor; uygun CUDA nvcc kurulmaya çalışılıyor...')
    try:
        install_cuda_nvcc()
        NVCC, ARCHS = pick_nvcc()
    except Exception as e:
        print('⚠️  CUDA kurulamadı, varsa PTX (JIT) yedeğine geçiliyor:', e)
if not NVCC:
    raise RuntimeError('nvcc bulunamadı. CUDA Toolkit kur (sm_120/Blackwell için ≥ 12.8): '
                       'https://developer.nvidia.com/cuda-downloads' +
                       ('' if IS_WIN else '  ya da root olarak çalıştırıp AUTO_INSTALL_CUDA=True bırak.'))
CUDA_ROOT = os.path.dirname(os.path.dirname(os.path.realpath(NVCC)))

if CC in ARCHS:
    GEN_VS = f'-gencode=arch=compute_{CC},code=sm_{CC} -gencode=arch=compute_{CC},code=compute_{CC}'
    print(f'✅ Yerel derleme: sm_{CC}  ({NVCC})')
elif any(a <= CC for a in ARCHS):
    ptx = max(a for a in ARCHS if a <= CC)
    GEN_VS = f'-gencode=arch=compute_{ptx},code=compute_{ptx}'
    print(f'⚠️  sm_{CC} desteklenmiyor → compute_{ptx} PTX üretilecek, sürücü ilk çalıştırmada JIT derleyecek.')
else:
    raise RuntimeError(f'{NVCC} bu GPU\'yu (sm_{CC}) desteklemiyor (en düşük arch {min(ARCHS)}); daha eski bir CUDA Toolkit kur.')

if IS_WIN:
    def find_vcvars():
        """MSVC ortamı: cl.exe zaten PATH'teyse gerek yok; yoksa vswhere ile vcvars64.bat."""
        if shutil.which('cl'): return None
        vswhere = os.path.expandvars(r'%ProgramFiles(x86)%\Microsoft Visual Studio\Installer\vswhere.exe')
        if os.path.exists(vswhere):
            out = sh([vswhere, '-latest', '-products', '*', '-requires', 'Microsoft.VisualStudio.Component.VC.Tools.x86.x64',
                      '-find', r'VC\Auxiliary\Build\vcvars64.bat'], check=False, quiet=True).strip().splitlines()
            if out and os.path.exists(out[-1]): return out[-1]
        c = sorted(glob.glob(os.path.expandvars(r'%ProgramFiles%\Microsoft Visual Studio\*\*\VC\Auxiliary\Build\vcvars64.bat')) +
                   glob.glob(os.path.expandvars(r'%ProgramFiles(x86)%\Microsoft Visual Studio\*\*\VC\Auxiliary\Build\vcvars64.bat')))
        if c: return c[-1]
        raise RuntimeError('MSVC bulunamadı: "Visual Studio 2022 Build Tools" içinden "C++ ile masaüstü geliştirme" bileşenini kur.')
    VCVARS = find_vcvars()
    print(f'  MSVC: {VCVARS or "cl.exe PATH içinde"}')

VS_DIR = os.path.join(WORK, 'VanitySearch-Blackwell')
VS_BIN = os.path.join(VS_DIR, 'vanitysearch.exe' if IS_WIN else 'vanitysearch')
JOBS = os.cpu_count() or 2

# Öz-test geçmezse dönülecek orijinal kaynak: depo geçmişindeki "Import VanitySearch-Bitcrack by FixedPaul" commit'i
VS_ORIGINAL_REF = '0bc3e9f529683a79c1b9667902d7b0287ff9436b'
VS_BUILT = None        # 'blackwell' ya da 'original'
VS_LABEL = 'VanitySearch'
VS_CACHE = None        # derlenen aracın önbellek kopyası (aynı commit + aynı derleyici ayarı = yeniden derleme yok)

def _cache_path(rev, original, cxx):
    key = hashlib.sha1(f'{rev}|{GEN_VS}|{NVCC}|{cxx}|{original}'.encode()).hexdigest()[:16]
    return os.path.join(WORK, 'bin_cache', key, os.path.basename(VS_BIN))

# Yalnızca orijinal kaynak için: aracın CPU tarafı (SHA-256/checksum) bazı GCC'lerle hatalı derlenip GPU'nun bulduğu
# DOĞRU anahtarı "wrong private key" diye reddedebiliyor. Bu satır o durumda da aday anahtarı yazdırır; doğrulamayı
# Python yapar. VanitySearch-Blackwell bu satırı zaten içeriyor.
VS_CAND_PATCH = 'fprintf(stdout, "  CandKey: %s\\n", k.GetBase16().c_str());'

def _compile_linux(cxx, gencode):
    inc = ' '.join(f'-I{d}' for d in (f'{CUDA_ROOT}/include', f'{CUDA_ROOT}/targets/x86_64-linux/include') if os.path.isdir(d))
    lib = ' '.join(f'-L{d}' for d in (f'{CUDA_ROOT}/lib64', f'{CUDA_ROOT}/lib', f'{CUDA_ROOT}/targets/x86_64-linux/lib')
                   if os.path.isdir(d))
    sh(f'make -C {VS_DIR} clean; rm -f {VS_BIN}', check=False, quiet=True)
    sh(f'make -C {VS_DIR} -j{JOBS} all CUDA={CUDA_ROOT} NVCC={NVCC} CXX={cxx} CXXCUDA={shutil.which(cxx)} GENCODE="{gencode}" '
       f'CXXFLAGS="-mssse3 -Wno-write-strings -O2 -fno-strict-aliasing -I. {inc} -include cstdint" '
       f'LFLAGS="-lpthread {lib} -lcudart_static -ldl -lrt"', quiet=True, tail=8000)

def _compile_windows(gencode):
    # bench/build.py ile aynı: tek nvcc çağrısı, C++ tarafını MSVC (cl.exe) derler
    mk = open(os.path.join(VS_DIR, 'Makefile'), encoding='utf-8', errors='replace').read()
    src = re.search(r'^SRC\s*=\s*((?:.*\\\r?\n)*.*)', mk, re.M).group(1).replace('\\', ' ').split() + ['GPU/GPUEngine.cu']
    if os.path.exists(VS_BIN): os.remove(VS_BIN)
    bat = os.path.join(VS_DIR, 'build_win.bat')
    with open(bat, 'w', encoding='utf-8') as f:
        f.write('@echo off\n' + (f'call "{VCVARS}" >nul\n' if VCVARS else '') + f'cd /d "{VS_DIR}"\n'
                f'"{NVCC}" -O3 -m64 {gencode} --threads 0 -DWIN32 -DWIN64 -DNDEBUG -D_CRT_SECURE_NO_WARNINGS -D_CONSOLE '
                f'-Xcompiler "/O2 /EHsc /W0" -I. -lcudart_static -ladvapi32 -o vanitysearch.exe ' +
                ' '.join(src) + '\n')
    r = subprocess.run(['cmd', '/c', bat], capture_output=True, text=True, encoding='utf-8', errors='replace')
    if r.returncode or not os.path.exists(VS_BIN):
        print((r.stdout + r.stderr)[-8000:])
        raise RuntimeError(f'nvcc/MSVC derlemesi başarısız (rc={r.returncode})')

def fetch_source(ref):
    """VS_REPO'yu VS_DIR'e klonlar (ya da günceller) ve ref'i (dal, etiket, commit) temiz olarak çıkarır."""
    if os.path.isdir(VS_DIR) and not os.path.isdir(os.path.join(VS_DIR, '.git')):
        shutil.rmtree(VS_DIR)                                 # yarım kalmış bir klon
    if os.path.isdir(VS_DIR):
        origin = sh(['git', '-C', VS_DIR, 'remote', 'get-url', 'origin'], check=False, quiet=True).strip()
        if origin != VS_REPO:
            shutil.rmtree(VS_DIR)                             # başka bir kaynağın klonu (ör. notebook'un eski sürümü)
    if not os.path.isdir(VS_DIR):
        # autocrlf=false: Windows'ta da dosyalar depodaki satır sonlarıyla gelsin. Tam klon (~3 MB): orijinal
        # commit'e dönebilmek için geçmiş gerekli.
        sh(['git', 'clone', '-q', '-c', 'core.autocrlf=false', VS_REPO, VS_DIR], quiet=True)
    sh(['git', '-C', VS_DIR, 'fetch', '-q', '--tags', '--force', 'origin'], quiet=True)
    remote_branch = subprocess.run(['git', '-C', VS_DIR, 'rev-parse', '--verify', '-q', f'origin/{ref}'],
                                   capture_output=True).returncode == 0
    sh(['git', '-C', VS_DIR, 'checkout', '-q', '--force', '--detach', f'origin/{ref}' if remote_branch else ref], quiet=True)
    sh(['git', '-C', VS_DIR, 'clean', '-q', '-f', '-d', '-x'], quiet=True)    # eski derleme çıktıları, her derleme temiz
    return sh(['git', '-C', VS_DIR, 'rev-parse', '--short', 'HEAD'], quiet=True).strip()

def build_vs(cxx='g++', original=False):
    global VS_BUILT, VS_LABEL, VS_CACHE
    rev = fetch_source(VS_ORIGINAL_REF if original else VS_REF)
    VS_CACHE = _cache_path(rev, original, cxx)
    if os.path.exists(VS_CACHE):
        shutil.copy2(VS_CACHE, VS_BIN)
        print('  (bu commit bu ayarlarla daha önce derlenmiş: önbellekteki araç kullanılıyor)')
    else:
        _build(cxx, original)
        os.makedirs(os.path.dirname(VS_CACHE), exist_ok=True)
        shutil.copy2(VS_BIN, VS_CACHE)
    VS_BUILT = 'original' if original else 'blackwell'
    VS_LABEL = 'VanitySearch-Bitcrack (orijinal)' if original else 'VanitySearch-Blackwell'
    print(f'  {VS_LABEL} @ {rev}: ' + (sh([VS_BIN, '-v'], check=False, quiet=True).strip().splitlines() or ['?'])[-1])

def _build(cxx, original):
    if original:
        mk_path = os.path.join(VS_DIR, 'Makefile')
        mk = open(mk_path, encoding='utf-8').read()
        # Orijinal Makefile'daki sabit sm_60..sm_89 listesini bu GPU'nun mimarisiyle değiştir
        mk2 = re.sub(r'(?:-gencode=arch=compute_\d+,code=(?:sm|compute)_\d+\s*)+', '$(GENCODE) ', mk)
        if mk2 != mk:
            with open(mk_path, 'w', encoding='utf-8', newline='') as f: f.write(mk2)
        vc_path = os.path.join(VS_DIR, 'Vanity.cpp')
        vc = open(vc_path, encoding='utf-8', errors='surrogateescape', newline='').read()
        marker = 'fprintf(stdout, "  Endo:%d incr:%d comp:%d\\n", endomorphism, incr, mode);'
        if VS_CAND_PATCH not in vc:
            assert marker in vc, 'Vanity.cpp beklenen yerde değişmiş; aday anahtar satırı eklenemedi'
            with open(vc_path, 'w', encoding='utf-8', errors='surrogateescape', newline='') as f:
                f.write(vc.replace(marker, marker + '\n\t\t\t' + VS_CAND_PATCH, 1))
    if not IS_WIN and cxx != 'g++' and not shutil.which(cxx):
        pkg_install(cxx)
    for i, gencode in enumerate((GEN_VS, GEN_VS + ' -allow-unsupported-compiler')):
        if i: print('  → -allow-unsupported-compiler ile yeniden deneniyor (nvcc bu GCC/MSVC sürümünü tanımıyor olabilir)...')
        try:
            _compile_windows(gencode) if IS_WIN else _compile_linux(cxx, gencode)
            if os.path.exists(VS_BIN): break
        except RuntimeError as e:
            print(f'⚠️  {e}')
    assert os.path.exists(VS_BIN), 'vanitysearch ikilisi oluşmadı (yukarıdaki derleme çıktısına bak)'

t0 = time.time()
print(f'\n🔨 VanitySearch-Blackwell derleniyor ({VS_REPO} @ {VS_REF})...')
build_vs()
print(f'✅ {VS_LABEL} hazır ({time.time() - t0:.0f} s)')


In [ ]:
#@title 🧪 5) Öz-test: araç bu GPU'da gerçekten anahtar buluyor mu?
#@markdown Rastgele bir test anahtarı üretip adresini Python'da hesaplıyoruz, sonra GPU aracına 2^36'lık küçük bir aralıkta aratıyoruz.
#@markdown `multi` modunda adres, rastgele adreslerin arasına saklanıp `-i` ile verilir: gerçek aramanın kullandığı bloom filtresi yolu test edilir.
#@markdown Test geçmezse arama **başlamaz** (Blackwell'de bozuk çalışan bir araçla saatler yakmayalım).

if MULTI:
    ensure_address_file()         # listenin türü (P2PKH / BECH32) öz-testte de kullanılır

def selftest(mode, bits=36, timeout=300, decoys=1000):
    base = ((1 << 23) | secrets.randbelow(1 << 23)) << bits     # 2^bits'e hizalı başlangıç
    k = base + secrets.randbelow(1 << bits)
    addr = list_addr_from_key(k) if MULTI else addr_from_key(k)
    cmd = [VS_BIN, '-gpuId', str(GPU_ID), '-start', f'{base:x}', '-range', str(bits), '-stop'] + \
          (['-random'] if mode == 'native' else [])
    listfile = None
    if MULTI:
        # Hedef, listedeki türden rastgele adreslerin ortasına saklanır: araç bloom filtresiyle arar
        enc = (lambda h: p2wpkh_encode(h)) if MULTI_BECH32 else (lambda h: b58check_encode(b'\x00' + h))
        lines = [enc(secrets.token_bytes(20)) for _ in range(decoys)]
        lines.insert(decoys // 2, addr)
        listfile = os.path.join(WORK, 'selftest_list.txt')
        with open(listfile, 'w') as f: f.write('\n'.join(lines) + '\n')
        cmd += ['-i', listfile]
    else:
        cmd += [addr]
    t0, ok, own_check_broken, tail = time.time(), False, False, []
    for line in stream(cmd, cwd=VS_DIR, timeout=timeout):
        if not line: continue
        tail = (tail + [line])[-12:]
        if 'wrong private key' in line: own_check_broken = True
        ev = parse_line(line)
        if ev and ev[0] == 'keys' and k in ev[1]:
            ok = True; break
        if mode == 'blocks' and ('Range Finished' in line or 'KEY NOT FOUND' in line):
            break                                   # aralık bitti ama anahtar gelmedi
    if listfile: os.remove(listfile)
    kind = f' · -i listesi, {decoys + 1} adres, {"BECH32" if MULTI_BECH32 else "P2PKH"}' if MULTI else ''
    print(f'  {mode}{kind}: {"✅ GEÇTİ" if ok else "❌ BAŞARISIZ"}  ({time.time() - t0:.1f} s, test anahtarı {k:x})')
    if ok and own_check_broken:
        print('    (aracın kendi CPU adres kontrolü hatalı; anahtar GPU sonucundan alınıp Python ile doğrulandı)')
    if not ok:
        print('    son çıktı:\n    ' + '\n    '.join(tail))
    return ok

def _drop_cache():
    if VS_CACHE and os.path.exists(VS_CACHE): os.remove(VS_CACHE)   # testi geçemeyen derleme önbellekte kalmasın

SELFTEST = {'blocks': selftest('blocks', timeout=600)}   # ilk çalıştırma PTX-JIT ise uzun sürebilir
if not SELFTEST['blocks']: _drop_cache()
if not SELFTEST['blocks'] and VS_BUILT == 'blackwell':
    print('  → VanitySearch-Blackwell testi geçemedi; orijinal FixedPaul kaynağıyla yeniden derleniyor...')
    build_vs(original=True)
    SELFTEST['blocks'] = selftest('blocks')
    if not SELFTEST['blocks']: _drop_cache()
if not SELFTEST['blocks'] and CAN_APT:
    print('  → VanitySearch, yazarın Makefile\'ında sabitlediği g++-9 ile yeniden derleniyor...')
    try:
        build_vs('g++-9', original=True)
        SELFTEST['blocks'] = selftest('blocks')
        if not SELFTEST['blocks']: _drop_cache()
    except Exception as e:
        print('  g++-9 ile derlenemedi:', e)
if not SELFTEST['blocks']:
    raise RuntimeError('VanitySearch öz-testi geçemedi; bu GPU/derleme ile arama yapmak anlamsız.')

STRATEGY = 'blocks'
if VS_STRATEGY == 'native':
    if selftest('native', bits=34):
        STRATEGY = 'native'
    else:
        print('ℹ️  VanitySearch -random öz-testi geçmedi → rastgele-blok modunda kullanılacak.')
print(f"\n🏁 Strateji: {STRATEGY}  ·  {VS_LABEL}")


In [ ]:
#@title 🚀 6) ARAMA — durdurmak için ■ Stop. Tekrar çalıştırınca istatistikler Drive'dan devam eder.
if not MULTI and FOUND_KEY is not None and addr_from_key(FOUND_KEY) != TARGET_ADDRESS:
    FOUND_KEY = None      # önceki (ör. test) bulmacadan kalan sonuç; bu hedefle ilgisi yok

if MULTI:
    ensure_address_file()         # gerekirse adres listesini indir (tools/fetch_addresses.py)

def build_cmd(start=None, bits=None):
    target = ['-i', ADDRESS_FILE] if MULTI else ['-stop', TARGET_ADDRESS]
    if STRATEGY == 'native':
        rng = ['-start', f'{RANGE_START:x}', '-range', str(RANGE_BITS), '-random']
    else:
        rng = ['-start', f'{start:x}', '-range', str(bits)]
    return [VS_BIN, '-gpuId', str(GPU_ID), '-o', VS_OUT] + rng + target, VS_DIR

stats = load_stats()
stats['sessions'] += 1
done_blocks = load_done_blocks()
S = {'t0': time.time(), 'keys': 0, 'speed': 0.0, 'last': '', 'blocks': 0, 'restarts': 0, 'gpu': ''}
status = display(HTML('<i>başlıyor…</i>'), display_id=True)

def gpu_telemetry():
    q = sh(['nvidia-smi', '-i', str(GPU_ID), '--query-gpu=utilization.gpu,power.draw,temperature.gpu,clocks.sm',
            '--format=csv,noheader,nounits'], check=False, quiet=True).strip().split(',')
    return f'%{q[0].strip()} kullanım · {q[1].strip()} W · {q[2].strip()} °C · {q[3].strip()} MHz' if len(q) == 4 else ''

def render():
    if status is None:           # bir notebook ön yüzü yoksa (düz python) display handle None olur
        return
    el = time.time() - S['t0']
    sp = S['speed'] * 1e6
    tot = stats['total_keys']
    if STRATEGY == 'blocks':
        # bloklar tekrar taranmaz: kapsama, biten blokların toplamıdır
        covered = (len(done_blocks) << BLOCK_BITS) / RANGE_SIZE
        cov = ('Kapsama (biten bloklar)', fmt_prob(covered))
        day = min(1.0, sp * 86400 / max(1, RANGE_SIZE * (1 - covered)))
    else:
        # -random tekrarlı örnekler: N anahtar denendiyse beklenen kapsama 1 - e^(-N/aralık)
        cov = ('Kapsama (tahmini)', fmt_prob(-math.expm1(-tot / RANGE_SIZE)))
        day = -math.expm1(-sp * 86400 / RANGE_SIZE)
    rows = [
        ('Mod', f"{'multi: ' + ADDRESS_LIST if MULTI else 'puzzle'} / {VS_LABEL} / {STRATEGY}"),
        ('Hız', f'<b>{sp / 1e9:.3f} Gkey/s</b>'),
        ('GPU', S['gpu']),
        ('Bu oturum', f"{fmt_num(S['keys'])} anahtar · {fmt_dur(el)} · {S['blocks']} blok bitti · {S['restarts']} yeniden başlatma"),
        ('Tüm oturumlar', f"{fmt_num(tot)} anahtar · {stats['sessions']} oturum · {fmt_dur(stats['seconds'] + time.time() - last_save)}"),
        cov,
    ]
    if MULTI:
        rows.append(('Bulunan', str(len(FOUND_KEYS))))
    else:
        rows.append(('1 günde bulma şansı', fmt_prob(day) if sp else '-'))
    rows.append(('Son satır', f"<code>{S['last'][:140]}</code>"))
    status.update(HTML('<table style="font-family:monospace;font-size:13px">' +
                       ''.join(f'<tr><td style="padding-right:14px"><b>{a}</b></td><td>{b}</td></tr>' for a, b in rows) +
                       '</table>'))

def scan_vs_out():
    # VanitySearch bulduğunu -o dosyasına da yazar; ekrandan kaçırılırsa buradan yakalarız
    if not os.path.exists(VS_OUT): return False
    hit = False
    for line in open(VS_OUT, encoding='utf-8', errors='replace'):
        ev = parse_line(line)
        if ev and ev[0] == 'keys' and on_found(ev[1], 'vanitysearch -o dosyası'):
            hit = True
    return hit

def pick_block():
    nblocks = RANGE_SIZE >> BLOCK_BITS
    if len(done_blocks) >= nblocks:
        return None                               # tüm bloklar tarandı
    while True:
        idx = secrets.randbelow(nblocks)
        if idx not in done_blocks: return idx

last_save = last_render = last_tel = time.time()
quick_fails = 0           # GPU'ya hiç ulaşmadan art arda kapanan çalıştırmalar
if not MULTI and FOUND_KEY is not None:
    print('‼️  Bu bulmacanın anahtarı daha önce bulunmuş (yukarıdaki FOUND dosyası); arama başlatılmadı.')
log(f"Oturum başladı: {'multi/' + ADDRESS_LIST if MULTI else 'puzzle'} · {VS_LABEL}/{STRATEGY}")
try:
    scan_vs_out()
    while MULTI or FOUND_KEY is None:
        idx = None
        if STRATEGY == 'blocks':
            idx = pick_block()
            if idx is None:
                log('Tüm bloklar tarandı.'); break
            cmd, cwd = build_cmd(RANGE_START + (idx << BLOCK_BITS), BLOCK_BITS)
        else:
            cmd, cwd = build_cmd()
        run_keys, completed, last_progress, info = 0, False, time.time(), {}
        gpu_running = False       # ilk hız satırına kadar araç adres listesini yüklüyor / başlangıç anahtarlarını kuruyor
        run_tail = []
        for line in stream(cmd, cwd=cwd, info=info):
            now = time.time()
            if line:
                S['last'] = line
                run_tail = (run_tail + [line])[-15:]
                if not gpu_running:
                    last_progress = now   # yükleme ilerlemesi de canlılık sayılır
                ev = parse_line(line)
                if ev and ev[0] == 'keys':
                    if on_found(ev[1], os.path.basename(cmd[0])) and not MULTI: break
                elif ev and ev[0] == 'speed':
                    delta = max(0, ev[2] - run_keys); run_keys = ev[2]
                    S['keys'] += delta; stats['total_keys'] += delta
                    if 0 < ev[1] < 1e7: S['speed'] = ev[1]
                    last_progress, gpu_running = now, True
                elif 'Range Finished' in line or 'KEY NOT FOUND' in line:
                    completed = True
                elif re.search(r'error|wrong|fail|warning', line, re.I):
                    log(f'araç: {line}')
            limit = 180 if gpu_running else 1800   # yükleme aşamasında çıktısız uzun adımlar olabilir (sıralama, bloom)
            if now - last_progress > limit:
                log(f'⚠️  {limit} sn ilerleme yok, araç yeniden başlatılıyor.'); break
            if now - last_tel > 15: S['gpu'] = gpu_telemetry(); last_tel = now
            if now - last_render > 3: render(); last_render = now
            if now - last_save > 60:
                stats['seconds'] += now - last_save; save_stats(stats); last_save = now
        # puzzle: zaten bulduysak VS_OUT'u yeniden tarama (aynı anahtarı iki kez yazmasın)
        found_now = (not MULTI and FOUND_KEY is not None) or scan_vs_out()
        if not MULTI and found_now: break
        if idx is not None and completed:
            done_blocks.add(idx); mark_block_done(idx); S['blocks'] += 1
        elif not completed:
            S['restarts'] += 1
            quick_fails = 0 if gpu_running else quick_fails + 1
            if quick_fails >= 5:
                # Aynı hata tekrar tekrar (GPU hatası, bozuk liste, ...): sonsuz döngüye girme
                print('    son çıktı:\n    ' + '\n    '.join(run_tail))
                raise RuntimeError('Araç 5 kez art arda GPU aramasına başlamadan kapandı; yukarıdaki çıktıya bak.')
            log(f"araç kapandı (rc={info.get('rc')}), 5 sn sonra yeniden başlıyor.")
            time.sleep(5)
except KeyboardInterrupt:
    log('⏹️  Kullanıcı durdurdu.')
finally:
    stats['seconds'] += time.time() - last_save; last_save = time.time()
    save_stats(stats); render()
    if MULTI:
        print(f'\n{len(FOUND_KEYS)} anahtar bulundu. Dosyalar: {FOUND_PATHS}')
    elif FOUND_KEY is not None:
        print(f'\n🎉 ANAHTAR: {FOUND_KEY:064x}\n   Dosyalar: {FOUND_PATHS}\n   Aşağıdaki "Bulursan" bölümünü OKU.')

## 🎉 Bulursan ne yapmalı (ÇOK ÖNEMLİ)

**Fonları normal bir cüzdandan, açık mempool üzerinden göndermeye çalışma.** Bu adresten harcama yaptığın anda public key mempool'da görünür. Anahtar yalnızca 70 bitlik bir aralıkta olduğu için botlar Kangaroo ile özel anahtarı **saniyeler içinde** bulur. Sonra daha yüksek ücretli bir işlemle (RBF) senin işleminin yerine geçip fonları kendilerine çekerler. Bu olay 2024'te #66 bulmacasını çözen kişinin başına geldi.

1. Önce `FOUND_KEY.txt` dosyasını **çevrimdışı** bir yere yedekle: kağıda yaz, şifreli bir USB'ye kopyala.
2. İşlemi çevrimdışı bir makinede oluştur ve imzala. Örneğin Electrum'un offline modu ya da kendi kodun olabilir.
3. İşlemi **doğrudan bir madenciye özel kanaldan** gönder. Örnek: MARA Slipstream gibi, işlemi açık mempool'a koymadan bloğa ekleyen servisler. Böylece public key, işlem bir bloğa girene kadar kimseye görünmez.
4. Fonlar güvendeki adresine geçtikten sonra kayıt klasöründeki (Colab'da Drive) `FOUND_KEY.txt`'i ve yedek `FOUND_KEY_puzzle*.txt`'i ve notebook çıktısını sil. Bu çıktıyı kimseyle paylaşma.

### Notlar
* **Oturum koparsa:** Bulunan anahtar zaten kayıt klasörüne (Colab'da Drive) yazıldı. Arama hücresini (6) tekrar çalıştırman yeterli; toplam istatistik oradaki `stats.json`'dan devam eder.
* **`blocks` stratejisi:** Tamamlanan bloklar `blocks_done_b42.txt`'e yazılır ve bir daha taranmaz. `native` modda VanitySearch'ün kendi rastgeleliği kullanılır. 2^70'lik bir alanda tekrar tarama olasılığı ihmal edilebilir düzeydedir.
* **Tüm GPU gücü:** VanitySearch bütün SM'leri kullanır. Arama hücresinin altında anlık kullanım, güç, sıcaklık ve saat değerleri görünür.